# سبق 05 - ایجنٹک RAG


## ترتیب

یہ نوٹ بک Microsoft Agent Framework استعمال کرتے ہوئے Agentic RAG (Retrieval-Augmented Generation) پیٹرن کی مثال پیش کرتی ہے۔

**ضروریات:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — آپ کی Azure AI سرچ سروس کا اینڈپوائنٹ
- `AZURE_SEARCH_API_KEY` — آپ کی Azure AI سرچ API کلید
- ماحول کے متغیرات کے ذریعے ترتیب دیا گیا Azure OpenAI تعیناتی
- Azure CLI کی توثیق شدہ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ایجنٹک RAG کیا ہے؟

روایتی RAG ایک مقررہ طریقہ کار پر عمل کرتا ہے: دستاویزات حاصل کریں، پھر جواب تیار کریں۔ **ایجنٹک RAG** آگے بڑھتا ہے اور ایجنٹ کو خود مختاری دیتا ہے کہ وہ فیصلہ کرے **کب** اور **کیسے** معلومات حاصل کریں۔

ایجنٹک RAG کے ساتھ، ایجنٹ کر سکتا ہے:
- **فیصلہ کرنا** کہ جواب دینے سے پہلے معلومات حاصل کرنا ضروری ہے یا نہیں
- **چننا** کہ کس ڈیٹا سورس یا ٹول سے معلومات حاصل کرنی ہے
- **جانچنا** حاصل کردہ نتائج اور اگر پہلا کوشش ناکافی ہو تو مزید معلومات حاصل کرنا
- **معلومات کو یکجا کرنا** مختلف حصول کے مراحل سے حاصل کردہ معلومات کو ایک مربوط جواب میں شامل کرنا

اس سے ایجنٹ ایک جامد حصول-اس کے بعد تیار کرنے والے طریقہ کار کے مقابلے میں زیادہ لچکدار اور درست ہو جاتا ہے۔


## سرچ ٹول بنانا

ایجنٹک RAG میں، بیرونی ڈیٹا ذرائع کو **ٹولز** کے طور پر لپیٹ کر رکھا جاتا ہے جنہیں ایجنٹ ضرورت کے مطابق بلا سکتا ہے۔ اس سے ایجنٹ کو بازیافت کو صرف ایک اور عمل کے طور پر دیکھنے کی اجازت ملتی ہے جو وہ انجام دے سکتا ہے، نہ کہ ایک لازمی قدم۔

نیچے ہم ایک سفری نالج بیس کی تعریف کرتے ہیں اور اسے ایک ٹول کے طور پر پیش کرتے ہیں جسے ایجنٹ منزل کی معلومات تلاش کرنے کے لیے بلا سکتا ہے۔


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## رَیگ ایجنٹ تیار کرنا

اب ہم ایک ایجنٹ تیار کرتے ہیں جسے ہمیشہ جواب دینے سے پہلے **معلومات حاصل کرنے کی ہدایت دی جاتی ہے**۔ یہ ایجنٹ `search_travel_knowledge` ٹول کا استعمال کرتا ہے تاکہ اپنے جوابات کو علم کے ذخیرے کی بنیاد پر رکھے، بجائے اس کے کہ وہ اپنے تربیتی ڈیٹا پر انحصار کرے۔


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## تکراری بازیافت — میکر-چیکر پیٹرن

ایجنٹک RAG کا ایک اہم فائدہ **تکراری بازیافت** ہے۔ ایجنٹ کئی بار تلاش کر سکتا ہے تاکہ اپنی ابتدائی معلومات کی تصدیق، بہترکاری، یا توسیع کرے — بالکل ایک "میکر-چیکر" کام کے بہاؤ کی طرح:

1. **میکر مرحلہ**: ایجنٹ ابتدائی معلومات حاصل کرتا ہے اور ایک جواب کا مسودہ تیار کرتا ہے۔
2. **چیکر مرحلہ**: ایجنٹ اضافی بازیافتیں کرتا ہے تاکہ تفصیلات کی تصدیق کرے یا خالی جگہیں پر کرے۔

نیچے، ایجنٹ سے ایک ایسا سوال پوچھا گیا ہے جس میں متعدد مقامات کا موازنہ کرنا ہوتا ہے، جس سے اسے کئی بار تلاش کرنے کی ترغیب ملتی ہے۔


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ مائیکروسافٹ ایجنٹ فریم ورک کا استعمال کرتے ہوئے **Agentic RAG** نظام کیسے بنایا جاتا ہے:

- **Agentic RAG** ایجنٹوں کو خودکار طریقے سے فیصلہ کرنے دیتا ہے کہ کب معلومات حاصل کرنی ہے، جس سے حاصل کرنے کا عمل متحرک ہو جاتا ہے بجائے کہ مقررہ۔
- **اوزار بطور ڈیٹا ذرائع**: خارجی معلوماتی بنیادیں (جیسے Azure AI سرچ) اوزار کے طور پر لپٹی ہوئی ہیں جنہیں ایجنٹ استعمال کر سکتا ہے۔
- **تکراری حصول**: میکر-چیکر پیٹرن ایجنٹ کو متعدد حصولی راونڈز انجام دینے کے قابل بناتا ہے — تلاش، تصدیق، اور ترمیم — آخری جواب دینے سے پہلے۔

پیداوار میں، آپ میموری میں موجود `TRAVEL_KNOWLEDGE_BASE` کو ایک حقیقی Azure AI سرچ انڈیکس سے بدل دیں گے تاکہ بڑے پیمانے پر سفر کے دستاویزات کی بازیافت کو سنبھالا جا سکے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
